# Feature Engineering & Preprocessing Pipeline

Merges `members`, `transactions`, and `user_logs` into one row per user, engineers the forward-revenue target and behavioral features, then encodes/scales/splits the data so it's ready for the PyTorch `Dataset` in the next notebook.

**Naming note**: the revenue target is called **forward revenue** (`fwd_rev_59d`), not "LTV" — it's observed revenue over a fixed 59-day forward window (two billing cycles), not an estimate of customer lifetime value. See `project_report.md` section 3.2 for the full argument.

**Survivorship bias fix**: earlier versions of this pipeline used KKBox's official `train`/`train_v2` label files, which only include users whose subscription happened to be expiring in one fixed snapshot window (Jan/Feb 2017). That silently excludes anyone who churned earlier in the 2015-2016 history and never came back — about 40% of everyone who ever paid KKBox (see `01_EDA.ipynb` discussion). This version instead builds its own population and labels directly from the full `transactions` history: every user's reference date (`ref_date`) is *their own* last observable paid subscription cycle, not a shared global date. A user who churned for good in mid-2015 is now included, using their own 2015 cycle as the reference point, on equal footing with someone still subscribed in late 2016.

Two consequences worth flagging up front:
- **Free trials are excluded from being a reference cycle** — only real (`actual_amount_paid > 0`) transactions establish a `ref_date`, so trial-and-done users don't count as "customer churn."
- **This changes the target rate substantially**: measured across full customer lifetimes instead of a survivors-only snapshot, churn goes from ~9% to ~74%. That's the honest picture the old snapshot was structurally blind to, not a data error.

In [ ]:
import json
import os

import duckdb
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
con = duckdb.connect()


def p(name):
    return os.path.join(PROCESSED_DIR, f"{name}.parquet")


# Per-user reference date replaces the old single global FEATURE_CUTOFF to remove
# survivorship bias: each user's own reference cycle is the latest PAID subscription
# expiration in their own history, not a shared snapshot date. Every feature and label
# below is computed relative to that user's own ref_date, not a global one.
GLOBAL_MAX_DATE = pd.Timestamp("2017-02-28")  # last date covered by transactions/user_logs
FWD_REV_WINDOW_DAYS = 59  # forward-revenue window length (two ~30-day billing cycles - see
                          # project_report.md section 3.2; matches the original
                          # Jan1-Feb28 span, but that's a coincidence of the Kaggle data's
                          # span, not the reason for 59)
CHURN_GRACE_DAYS = 30     # KKBox's own definition: no renewal within 30 days of expiry = churn
ENGAGEMENT_WINDOW_DAYS = 30  # trailing listening-activity window ending at each user's ref_date

# Latest a user's reference cycle can end and still leave a full FWD_REV_WINDOW_DAYS of real
# future data to observe (evaluates to 2016-12-31 -- same date the old global cutoff used,
# for the same reason: it's the latest point that still leaves 59 days of runway before
# the data ends on 2017-02-28).
REF_DATE_CUTOFF = GLOBAL_MAX_DATE - pd.Timedelta(days=FWD_REV_WINDOW_DAYS)
print(f"REF_DATE_CUTOFF = {REF_DATE_CUTOFF.date()}")

## 1. Data merging strategy

Base population: every `msno` with at least one real (`actual_amount_paid > 0`) transaction whose `membership_expire_date` is on/before `REF_DATE_CUTOFF`. That transaction's expiry becomes the user's `ref_date`. All labels and features below are computed relative to each row's own `ref_date`, via a join against a per-user `ref_dates` table rather than a single global date filter.

In [ ]:
con.execute(f"""
    create or replace temp table txn as
    select *,
           strptime(cast(transaction_date as varchar), '%Y%m%d')::date as txn_dt,
           strptime(cast(membership_expire_date as varchar), '%Y%m%d')::date as expire_dt
    from '{p("transactions")}'
""")

con.execute(f"""
    create or replace temp table ref_dates as
    -- each user's own last PAID cycle that still leaves a full FWD_REV_WINDOW_DAYS of
    -- observable future data; free-trial ($0) transactions don't establish a ref_date
    select msno, max(expire_dt) as ref_date
    from txn
    where expire_dt <= date '{REF_DATE_CUTOFF.date()}' and actual_amount_paid > 0
    group by msno
""")
n_ref = con.execute("select count(*) from ref_dates").fetchone()[0]
print(f"users with a valid reference cycle: {n_ref:,}")

RECENT_ENGAGEMENT_WINDOW_DAYS = 7  # short trailing window, compared against ENGAGEMENT_WINDOW_DAYS for a trend signal

merge_query = f"""
    with churn_flag as (
        -- KKBox's own definition: churn = no renewal transaction within CHURN_GRACE_DAYS of expiry
        select r.msno,
               case when exists (
                   select 1 from txn t
                   where t.msno = r.msno and t.txn_dt > r.ref_date
                     and t.txn_dt <= r.ref_date + interval {CHURN_GRACE_DAYS} day
               ) then 0 else 1 end as is_churn
        from ref_dates r
    ),
    fwd_rev_target as (
        -- forward-revenue target: forward-looking revenue strictly after each user's own ref_date
        select t.msno, sum(t.actual_amount_paid) as fwd_rev_59d
        from txn t join ref_dates r using (msno)
        where t.txn_dt > r.ref_date and t.txn_dt <= r.ref_date + interval {FWD_REV_WINDOW_DAYS} day
        group by t.msno
    ),
    txn_agg as (
        -- feature aggregates: transactions on/before each user's own ref_date only
        select t.msno,
               count(*) as num_transactions,
               avg(t.payment_plan_days) as avg_payment_plan_days,
               avg(t.actual_amount_paid) as avg_actual_amount_paid,
               avg(t.is_auto_renew) as is_auto_renew_rate,
               avg(t.is_cancel) as is_cancel_rate,
               -- plan_list_price=0 on ~7% of all transactions (data quality artifact, not a
               -- real free plan - these are excluded from the ratio, same treatment as the
               -- corrupted membership_expire_date / total_secs guards elsewhere in this
               -- notebook); avg() ignores the NULLs this produces for excluded rows.
               avg(case when t.plan_list_price > 0
                        then (t.plan_list_price - t.actual_amount_paid) / t.plan_list_price
                   end) as avg_discount_rate
        from txn t join ref_dates r using (msno)
        where t.txn_dt <= r.ref_date
        group by t.msno
    ),
    payment_method_count as (
        -- distinct payment methods used across each user's pre-ref_date transaction history
        select t.msno, count(distinct t.payment_method_id) as num_distinct_payment_methods
        from txn t join ref_dates r using (msno)
        where t.txn_dt <= r.ref_date
        group by t.msno
    ),
    latest_txn as (
        -- most recent pre-ref_date transaction snapshot: payment method plus plan/amount/
        -- auto-renew/cancel "as of right now" - a sharper, distinct signal from txn_agg's
        -- lifetime averages above. Validated empirically (see 03a_CatBoost_and_Cox_Models.ipynb):
        -- most_recent_is_cancel alone came out as the single most important forward-revenue
        -- feature, well ahead of the lifetime is_cancel_rate - recency beats average here.
        select msno, payment_method_id, payment_plan_days as most_recent_payment_plan_days,
               actual_amount_paid as most_recent_actual_amount_paid,
               is_auto_renew as most_recent_is_auto_renew, is_cancel as most_recent_is_cancel
        from (
            select t.msno, t.payment_method_id, t.payment_plan_days, t.actual_amount_paid,
                   t.is_auto_renew, t.is_cancel,
                   row_number() over (partition by t.msno order by t.txn_dt desc) rn
            from txn t join ref_dates r using (msno)
            where t.txn_dt <= r.ref_date
        ) where rn = 1
    ),
    last_login as (
        -- unbounded lookback (unlike logs_agg's 30-day window below) - recency of the most
        -- recent listening activity, a different signal from "how much activity in 30 days"
        select l.msno, max(l.log_dt) as last_login_dt
        from (
            select *, strptime(cast(date as varchar), '%Y%m%d')::date as log_dt
            from '{p("user_logs")}'
        ) l
        join ref_dates r using (msno)
        where l.log_dt <= r.ref_date
        group by l.msno
    ),
    last_cancel as (
        -- most recent cancellation transaction date - distinct from is_cancel_rate's
        -- lifetime average rate
        select t.msno, max(t.txn_dt) as last_cancel_dt
        from txn t join ref_dates r using (msno)
        where t.txn_dt <= r.ref_date and t.is_cancel = 1
        group by t.msno
    ),
    logs_agg as (
        -- total_secs clipped to [0, 86400]: a small fraction of rows have corrupted
        -- extreme values (see 01_EDA.ipynb, "Scanning the full user_logs history" section).
        -- Single pass over user_logs (392M rows): the 30-day window is the outer filter,
        -- and the 7-day recent-window columns use conditional aggregates on the same
        -- scan instead of extra joins, to avoid doubling I/O over the largest table.
        select l.msno,
               count(distinct l.log_dt) as daily_active_days,
               -- (msno, date) is unique in user_logs (verified: one row per user per day),
               -- so a plain conditional count is equivalent to count(distinct ...) here but
               -- far cheaper - no second per-group hash-set needed across 392M rows.
               count(*) filter (where l.log_dt >= r.ref_date - interval {RECENT_ENGAGEMENT_WINDOW_DAYS - 1} day)
                   as recent_daily_active_days,
               sum(greatest(least(l.total_secs, 86400), 0)) as total_secs_sum,
               sum(l.num_25) as sum25, sum(l.num_50) as sum50, sum(l.num_75) as sum75,
               sum(l.num_985) as sum985, sum(l.num_100) as sum100,
               sum(l.num_unq) as sum_unq,
               -- recent-7-days vs prior-23-days raw sums, feeding the secs/numunq trend
               -- ratios computed in pandas below (magnitude-based trend, distinct from
               -- recent_engagement_ratio's coarser active-day-count-only signal)
               sum(greatest(least(l.total_secs, 86400), 0))
                   filter (where l.log_dt >= r.ref_date - interval 6 day) as recent7_secs,
               sum(greatest(least(l.total_secs, 86400), 0))
                   filter (where l.log_dt < r.ref_date - interval 6 day) as prior23_secs,
               sum(l.num_unq) filter (where l.log_dt >= r.ref_date - interval 6 day) as recent7_numunq,
               sum(l.num_unq) filter (where l.log_dt < r.ref_date - interval 6 day) as prior23_numunq
        from (
            select *, strptime(cast(date as varchar), '%Y%m%d')::date as log_dt
            from '{p("user_logs")}'
        ) l
        join ref_dates r using (msno)
        where l.log_dt >= r.ref_date - interval {ENGAGEMENT_WINDOW_DAYS - 1} day
          and l.log_dt <= r.ref_date
        group by l.msno
    )
    select
        r.msno, r.ref_date, cf.is_churn,
        m.city, m.bd, m.gender, m.registered_via, m.registration_init_time,
        coalesce(fr.fwd_rev_59d, 0) as fwd_rev_59d,
        coalesce(txn_agg.num_transactions, 0) as num_transactions,
        txn_agg.avg_payment_plan_days,
        txn_agg.avg_actual_amount_paid,
        coalesce(txn_agg.is_auto_renew_rate, 0) as is_auto_renew_rate,
        coalesce(txn_agg.is_cancel_rate, 0) as is_cancel_rate,
        coalesce(txn_agg.avg_discount_rate, 0) as avg_discount_rate,
        coalesce(payment_method_count.num_distinct_payment_methods, 0) as num_distinct_payment_methods,
        latest_txn.payment_method_id,
        latest_txn.most_recent_payment_plan_days,
        latest_txn.most_recent_actual_amount_paid,
        coalesce(latest_txn.most_recent_is_auto_renew, 0) as most_recent_is_auto_renew,
        coalesce(latest_txn.most_recent_is_cancel, 0) as most_recent_is_cancel,
        date_diff('day', ll.last_login_dt, r.ref_date) as days_since_last_login,
        date_diff('day', lc.last_cancel_dt, r.ref_date) as days_since_last_cancel,
        coalesce(logs_agg.daily_active_days, 0) as daily_active_days,
        coalesce(logs_agg.recent_daily_active_days, 0) as recent_daily_active_days,
        coalesce(logs_agg.total_secs_sum, 0) as total_secs_sum,
        coalesce(logs_agg.sum25, 0) as sum25, coalesce(logs_agg.sum50, 0) as sum50,
        coalesce(logs_agg.sum75, 0) as sum75, coalesce(logs_agg.sum985, 0) as sum985,
        coalesce(logs_agg.sum100, 0) as sum100,
        coalesce(logs_agg.sum_unq, 0) as sum_unq,
        coalesce(logs_agg.recent7_secs, 0) as recent7_secs,
        coalesce(logs_agg.prior23_secs, 0) as prior23_secs,
        coalesce(logs_agg.recent7_numunq, 0) as recent7_numunq,
        coalesce(logs_agg.prior23_numunq, 0) as prior23_numunq
    from ref_dates r
    left join churn_flag cf using (msno)
    left join '{p("members")}' m using (msno)
    left join txn_agg using (msno)
    left join payment_method_count using (msno)
    left join latest_txn using (msno)
    left join last_login ll using (msno)
    left join last_cancel lc using (msno)
    left join fwd_rev_target fr using (msno)
    left join logs_agg using (msno)
"""
df = con.execute(merge_query).df()
print(f"merged shape: {df.shape}")
print(f"churn rate: {df['is_churn'].mean():.4f}")
df.isna().sum()

1. `avg_payment_plan_days` / `avg_actual_amount_paid` / `payment_method_id` are now null only for a small number of rows (~3.2K) where the source `payment_plan_days` field itself is null on the transaction that produced `ref_date` — no longer the ~31.5K "no pre-cutoff transaction" case from the old global-cutoff version, since by construction every row now has at least one qualifying transaction (the one that set its own `ref_date`).
2. `city`/`bd`/`gender`/`registered_via`/`registration_init_time` are null for users absent from `members.parquet` entirely.
3. `gender` is additionally null (missing but present in `members`) for many more, consistent with the ~65% gender-missing rate found in `01_EDA.ipynb`.

## 5.2 Feature engineering

In [3]:
# Registration tenure (days): each user's own ref_date - registration_init_time
reg_date = pd.to_datetime(
    df["registration_init_time"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce"
)
df["registration_tenure_days"] = (pd.to_datetime(df["ref_date"]) - reg_date).dt.days

# Weighted song completion rate
song_totals = df[["sum25", "sum50", "sum75", "sum985", "sum100"]].sum(axis=1)
df["avg_song_completion"] = (
    0.25 * df["sum25"] + 0.50 * df["sum50"] + 0.75 * df["sum75"] + 0.985 * df["sum985"] + 1.0 * df["sum100"]
) / (song_totals + 1)

# Log listening time (30-day pre-ref_date window) and log-forward-revenue target
df["total_secs_log"] = np.log1p(df["total_secs_sum"])
df["log1p_fwd_rev_59d"] = np.log1p(df["fwd_rev_59d"])

# Listening diversity: unique songs played per active day (distinct from the completion-rate
# signal above, which measures how much of each song gets played, not how varied the songs are)
df["avg_num_unq_songs"] = df["sum_unq"] / (df["daily_active_days"] + 1)

# Engagement trend: recent (7-day) vs longer (30-day) active-day rate, ahead of ref_date.
# A ratio well above the ~0.23 (7/30) baseline means engagement picked up recently; well below
# means it's tapering off - information the single 30-day average can't distinguish.
df["recent_engagement_ratio"] = df["recent_daily_active_days"] / (df["daily_active_days"] + 1)

# Magnitude-based listening trend: recent-7-day vs prior-23-day *intensity* (secs played,
# unique songs), normalized to a per-day rate. Distinct from recent_engagement_ratio above,
# which only measures whether a day was active at all, not how much happened on it - a user
# going from 3 songs/day to 30 songs/day looks identical to recent_engagement_ratio but very
# different here. Grounded in the WSDM 2018 winning solution's top features (see README).
df["secs_trend_recent_vs_prior"] = (df["recent7_secs"] / 7.0) / ((df["prior23_secs"] / 23.0) + 1)
df["numunq_trend_recent_vs_prior"] = (df["recent7_numunq"] / 7.0) / ((df["prior23_numunq"] / 23.0) + 1)

# txn_frequency (num_transactions normalized by tenure) is computed later, after
# registration_tenure_days is median-imputed - see the preprocessing checklist below.

df[[
    "registration_tenure_days", "avg_song_completion", "total_secs_log", "log1p_fwd_rev_59d",
    "avg_num_unq_songs", "recent_engagement_ratio",
    "is_cancel_rate", "avg_discount_rate", "num_distinct_payment_methods",
    "days_since_last_login", "days_since_last_cancel",
    "secs_trend_recent_vs_prior", "numunq_trend_recent_vs_prior",
    "most_recent_payment_plan_days", "most_recent_actual_amount_paid",
    "most_recent_is_auto_renew", "most_recent_is_cancel",
]].describe()

,registration_tenure_days,avg_song_completion,total_secs_log,log1p_fwd_rev_59d,avg_num_unq_songs,recent_engagement_ratio,is_cancel_rate,avg_discount_rate,num_distinct_payment_methods,days_since_last_login,days_since_last_cancel,secs_trend_recent_vs_prior,numunq_trend_recent_vs_prior,most_recent_payment_plan_days,most_recent_actual_amount_paid,most_recent_is_auto_renew,most_recent_is_cancel
count,1.418160e+06,1.610171e+06,1.610171e+06,1.610171e+06,1.610171e+06,1.610171e+06,1.610171e+06,1.610171e+06,1.610171e+06,1394670.0,473017.0,1.610171e+06,1.610171e+06,1606934.0,1606934.0,1.610171e+06,1.610171e+06
mean,1.202783e+03,6.280897e-01,8.517830e+00,3.160336e+00,1.786602e+01,1.521396e-01,4.350598e-02,1.428348e-02,1.186372e+00,19.617202,142.37553,1.024081e+01,7.172659e-01,51.362191,230.622356,6.960956e-01,1.566858e-01
std,1.068820e+03,3.605981e-01,4.814518e+00,2.565425e+00,1.969833e+01,1.328638e-01,1.090730e-01,6.513664e-02,4.698832e-01,68.184453,187.789663,2.115788e+02,1.334075e+00,78.241772,341.182736,4.599420e-01,3.635044e-01
min,-1.717400e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-8.859358e-02,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
25%,3.650000e+02,4.910741e-01,7.153570e+00,0.000000e+00,2.204167e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.0,13.0,0.000000e+00,0.000000e+00,30.0,99.0,0.000000e+00,0.000000e+00
50%,9.300000e+02,7.908688e-01,1.078058e+01,4.605170e+00,1.380000e+01,1.818182e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.0,29.0,5.339171e-01,4.766839e-01,30.0,149.0,1.000000e+00,0.000000e+00
75%,1.740000e+03,8.939672e-01,1.183673e+01,5.293305e+00,2.512500e+01,2.333333e-01,4.166667e-02,0.000000e+00,1.000000e+00,6.0,263.0,1.129174e+00,1.014109e+00,30.0,149.0,1.000000e+00,0.000000e+00
max,4.663000e+03,9.999356e-01,1.476593e+01,8.002694e+00,6.292258e+02,8.750000e-01,1.000000e+00,1.000000e+00,8.000000e+00,730.0,730.0,4.886018e+04,2.364286e+02,450.0,2000.0,1.000000e+00,1.000000e+00


## 5.3 Preprocessing checklist

Split first, then fit every encoder/scaler/imputer statistic **only on the training split** to avoid leakage

In [4]:
df["gender"] = df["gender"].fillna("unknown")
df["bd_clean"] = df["bd"].where(df["bd"].between(1, 100))  # invalid ages imputed after the split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["is_churn"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["is_churn"], random_state=42)
splits = {"train": train_df, "val": val_df, "test": test_df}

for name, split in splits.items():
    print(f"{name:6s} n={len(split):>7,}  churn_rate={split['is_churn'].mean():.4f}")

train  n=1,127,119  churn_rate=0.7360
val    n=241,526  churn_rate=0.7360
test   n=241,526  churn_rate=0.7360


In [5]:
# Age: median imputation fit on train only
bd_median = train_df["bd_clean"].median()
# Registration tenure: median imputation for the ~11.7% of users absent from members.csv
tenure_median = train_df["registration_tenure_days"].median()
# Transaction aggregates: median imputation for users with no pre-cutoff transaction history
payment_plan_days_median = train_df["avg_payment_plan_days"].median()
actual_amount_paid_median = train_df["avg_actual_amount_paid"].median()
# Most-recent-transaction snapshot: same ~3.2K-row null population as avg_payment_plan_days
# above (the transaction that set ref_date itself has a null payment_plan_days field) -
# same median-imputation treatment.
most_recent_plan_days_median = train_df["most_recent_payment_plan_days"].median()
most_recent_amount_paid_median = train_df["most_recent_actual_amount_paid"].median()
# Recency sentinels: "never logged in" / "never canceled" before ref_date are legitimate,
# common states (13.4% / 70.6% of users respectively) - not missing data to impute toward
# the middle. A sentinel larger than any real observed gap keeps them correctly ordered as
# "furthest away" for a tree-based split, without needing a separate boolean flag column
# (empirically confirmed: an explicit never_X flag added no value on top of the sentinel -
# see 03's feature-ablation notes).
login_sentinel = train_df["days_since_last_login"].max() + 30
cancel_sentinel = train_df["days_since_last_cancel"].max() + 30
print(f"bd median (train): {bd_median}, tenure median (train): {tenure_median}")
print(f"avg_payment_plan_days median (train): {payment_plan_days_median}, "
      f"avg_actual_amount_paid median (train): {actual_amount_paid_median}")
print(f"most_recent_payment_plan_days median (train): {most_recent_plan_days_median}, "
      f"most_recent_actual_amount_paid median (train): {most_recent_amount_paid_median}")
print(f"days_since_last_login sentinel (train max + 30): {login_sentinel}, "
      f"days_since_last_cancel sentinel (train max + 30): {cancel_sentinel}")

for split in splits.values():
    split["bd_clean"] = split["bd_clean"].fillna(bd_median)
    split["registration_tenure_days"] = split["registration_tenure_days"].fillna(tenure_median)
    split["avg_payment_plan_days"] = split["avg_payment_plan_days"].fillna(payment_plan_days_median)
    split["avg_actual_amount_paid"] = split["avg_actual_amount_paid"].fillna(actual_amount_paid_median)
    split["most_recent_payment_plan_days"] = split["most_recent_payment_plan_days"].fillna(most_recent_plan_days_median)
    split["most_recent_actual_amount_paid"] = split["most_recent_actual_amount_paid"].fillna(most_recent_amount_paid_median)
    split["days_since_last_login"] = split["days_since_last_login"].fillna(login_sentinel)
    split["days_since_last_cancel"] = split["days_since_last_cancel"].fillna(cancel_sentinel)
    # Transaction frequency normalized by tenure, computed only now that tenure has no NaNs -
    # distinguishes a long-tenured user with few transactions from a new user with few
    # transactions, which raw num_transactions can't. Negative tenure (data-quality artifact,
    # same as bd's garbage values) is clipped to 0 so the denominator is always >= 1.
    split["txn_frequency"] = split["num_transactions"] / (split["registration_tenure_days"].clip(lower=0) + 1)

bd median (train): 28.0, tenure median (train): 929.0
avg_payment_plan_days median (train): 30.0, avg_actual_amount_paid median (train): 149.0
most_recent_payment_plan_days median (train): 30.0, most_recent_actual_amount_paid median (train): 149.0
days_since_last_login sentinel (train max + 30): 760, days_since_last_cancel sentinel (train max + 30): 760


In [6]:
# Label-encode categoricals to integer indices starting at 0; an extra reserved index
# absorbs missing/unseen categories at val/test/inference time (nn.Embedding-safe).
CATEGORICAL_COLS = ["city", "gender", "registered_via", "payment_method_id"]
# Embedding dims per the plan's Section 3.1 heuristics
EMBEDDING_DIMS = {"city": 5, "gender": 2, "registered_via": 3, "payment_method_id": 8}


def fit_encoder(train_col):
    categories = sorted(train_col.dropna().unique().tolist())
    mapping = {cat: i for i, cat in enumerate(categories)}
    mapping["__unknown__"] = len(categories)
    return mapping


def apply_encoder(col, mapping):
    return col.map(mapping).fillna(mapping["__unknown__"]).astype(int)


encoders = {}
for col in CATEGORICAL_COLS:
    mapping = fit_encoder(train_df[col])
    encoders[col] = mapping
    for split in splits.values():
        split[f"{col}_enc"] = apply_encoder(split[col], mapping)
    print(f"{col:20s} cardinality (incl. unknown bucket): {len(mapping)}")

city                 cardinality (incl. unknown bucket): 22
gender               cardinality (incl. unknown bucket): 4


registered_via       cardinality (incl. unknown bucket): 7
payment_method_id    cardinality (incl. unknown bucket): 40


In [7]:
# StandardScaler on the unbounded numerical columns, fit on train only.
# is_auto_renew_rate/is_cancel_rate/avg_discount_rate/avg_song_completion/
# most_recent_is_auto_renew/most_recent_is_cancel are already in/near [0, 1]
# (avg_discount_rate can go slightly negative if a user ever paid above list price)
# and left unscaled.
SCALE_COLS = [
    "bd_clean", "registration_tenure_days", "avg_payment_plan_days",
    "avg_actual_amount_paid", "num_transactions", "total_secs_log", "daily_active_days",
    "num_distinct_payment_methods", "avg_num_unq_songs", "recent_engagement_ratio", "txn_frequency",
    "days_since_last_login", "days_since_last_cancel",
    "secs_trend_recent_vs_prior", "numunq_trend_recent_vs_prior",
    "most_recent_payment_plan_days", "most_recent_actual_amount_paid",
]
UNSCALED_NUMERICAL_COLS = [
    "is_auto_renew_rate", "avg_song_completion", "is_cancel_rate", "avg_discount_rate",
    "most_recent_is_auto_renew", "most_recent_is_cancel",
]

scaler = StandardScaler()
scaler.fit(train_df[SCALE_COLS])
for split in splits.values():
    split[[f"{c}_scaled" for c in SCALE_COLS]] = scaler.transform(split[SCALE_COLS])

train_df[[f"{c}_scaled" for c in SCALE_COLS]].describe()

,bd_clean_scaled,registration_tenure_days_scaled,avg_payment_plan_days_scaled,avg_actual_amount_paid_scaled,num_transactions_scaled,total_secs_log_scaled,daily_active_days_scaled,num_distinct_payment_methods_scaled,avg_num_unq_songs_scaled,recent_engagement_ratio_scaled,txn_frequency_scaled,days_since_last_login_scaled,days_since_last_cancel_scaled,secs_trend_recent_vs_prior_scaled,numunq_trend_recent_vs_prior_scaled,most_recent_payment_plan_days_scaled,most_recent_actual_amount_paid_scaled
count,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06
mean,-5.255066e-17,-1.185164e-16,3.792523e-17,2.178684e-17,-6.151503e-17,-1.679150e-16,-4.871779e-17,3.726961e-17,-1.428752e-16,5.650961e-17,2.420760e-17,-1.563407e-17,-1.786067e-16,-1.563407e-18,3.237766e-17,-2.793960e-17,1.603753e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-4.483852e+00,-1.689758e+01,-6.767010e-01,-7.090256e-01,-1.396991e+00,-1.769625e+00,-1.219097e+00,-2.525629e+00,-9.068536e-01,-1.145081e+00,-4.427541e-01,-4.563853e-01,-1.936242e+00,-4.815411e-02,-5.410110e-01,-6.563548e-01,-6.760205e-01
25%,-1.198030e-01,-7.647417e-01,-2.718738e-01,-3.478391e-01,-9.047752e-01,-2.829297e-01,-1.127596e+00,-3.961921e-01,-7.926138e-01,-1.145081e+00,-3.091124e-01,-4.563853e-01,-6.150614e-01,-4.815411e-02,-5.410110e-01,-2.728661e-01,-3.857042e-01
50%,-1.198030e-01,-2.394911e-01,-2.542726e-01,-2.274436e-01,-4.339702e-02,4.699923e-01,-2.958900e-02,-3.961921e-01,-2.061828e-01,2.229737e-01,-1.093683e-01,-4.525352e-01,6.057760e-01,-4.566928e-02,-1.815080e-01,-2.728661e-01,-2.390798e-01
75%,-1.198030e-01,4.237743e-01,-2.542726e-01,-2.274436e-01,6.949272e-01,6.892651e-01,9.769176e-01,-3.961921e-01,3.678221e-01,6.105893e-01,2.619023e-01,-3.716839e-01,6.057760e-01,-4.290105e-02,2.241101e-01,-2.728661e-01,-2.390798e-01
max,1.151766e+01,3.468043e+00,5.659725e+00,5.755162e+00,6.847629e+00,1.298021e+00,1.525921e+00,1.450987e+01,3.104098e+01,5.438683e+00,4.904679e+02,2.469663e+00,6.057760e-01,2.271642e+02,1.776648e+02,5.095975e+00,5.188955e+00


## Save model-ready splits, encoders, and feature manifest

Final feature set: 4 categorical + 23 numerical = 27 columns. Expanded twice from the original 13 (4 categorical + 9 numerical): first to 19 (adding `is_cancel_rate`, `avg_discount_rate`, `num_distinct_payment_methods`, `avg_num_unq_songs`, `recent_engagement_ratio`, `txn_frequency`), then to 27 with 8 more features grounded in the WSDM 2018 KKBox Churn Challenge winning solutions (see README): recency (`days_since_last_login`, `days_since_last_cancel`), most-recent-transaction snapshot (`most_recent_payment_plan_days`, `most_recent_actual_amount_paid`, `most_recent_is_auto_renew`, `most_recent_is_cancel` — validated as the single most important forward-revenue feature, ahead of the lifetime `is_cancel_rate`), and magnitude-based listening trend (`secs_trend_recent_vs_prior`, `numunq_trend_recent_vs_prior`). Two candidates from the same ablation round (`std_dev_daily_secs_30d`, and explicit `never_logged_in`/`never_canceled` boolean flags) were tested and found to add no value beyond what the sentinel-encoded recency features already capture — not included.

In [8]:
OUTPUT_COLS = (
    ["msno", "is_churn", "log1p_fwd_rev_59d", "fwd_rev_59d"]
    + [f"{c}_enc" for c in CATEGORICAL_COLS]
    + [f"{c}_scaled" for c in SCALE_COLS]
    + UNSCALED_NUMERICAL_COLS
)

for name, split in splits.items():
    out_path = os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet")
    split[OUTPUT_COLS].to_parquet(out_path, engine="pyarrow", compression="zstd", index=False)
    print(f"wrote {out_path} ({len(split):,} rows, {len(OUTPUT_COLS)} cols)")

joblib.dump(encoders, os.path.join(PROCESSED_DIR, "categorical_encoders.joblib"))
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "numerical_scaler.joblib"))

feature_manifest = {
    "methodology": "per-user event-based reference date (fixes survivorship bias vs. a single global snapshot cutoff)",
    "ref_date_max_cutoff": str(REF_DATE_CUTOFF.date()),
    "fwd_rev_window_days": FWD_REV_WINDOW_DAYS,
    "churn_grace_days": CHURN_GRACE_DAYS,
    "engagement_window_days": ENGAGEMENT_WINDOW_DAYS,
    "excludes_free_trial_ref_cycles": True,
    "targets": {"churn": "is_churn", "fwd_rev_log": "log1p_fwd_rev_59d", "fwd_rev_raw": "fwd_rev_59d"},
    "categorical": {
        col: {
            "column": f"{col}_enc",
            "cardinality": len(encoders[col]),
            "embedding_dim": EMBEDDING_DIMS[col],
            "unknown_index": encoders[col]["__unknown__"],
        }
        for col in CATEGORICAL_COLS
    },
    "numerical_scaled": [f"{c}_scaled" for c in SCALE_COLS],
    "numerical_unscaled": UNSCALED_NUMERICAL_COLS,
}
with open(os.path.join(PROCESSED_DIR, "feature_manifest.json"), "w") as f:
    json.dump(feature_manifest, f, indent=2)

feature_manifest

wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_train.parquet (1,127,119 rows, 31 cols)


wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_val.parquet (241,526 rows, 31 cols)


wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_test.parquet (241,526 rows, 31 cols)


{'methodology': 'per-user event-based reference date (fixes survivorship bias vs. a single global snapshot cutoff)',
 'ref_date_max_cutoff': '2016-12-31',
 'fwd_rev_window_days': 59,
 'churn_grace_days': 30,
 'engagement_window_days': 30,
 'excludes_free_trial_ref_cycles': True,
 'targets': {'churn': 'is_churn',
  'fwd_rev_log': 'log1p_fwd_rev_59d',
  'fwd_rev_raw': 'fwd_rev_59d'},
 'categorical': {'city': {'column': 'city_enc',
   'cardinality': 22,
   'embedding_dim': 5,
   'unknown_index': 21},
  'gender': {'column': 'gender_enc',
   'cardinality': 4,
   'embedding_dim': 2,
   'unknown_index': 3},
  'registered_via': {'column': 'registered_via_enc',
   'cardinality': 7,
   'embedding_dim': 3,
   'unknown_index': 6},
  'payment_method_id': {'column': 'payment_method_id_enc',
   'cardinality': 40,
   'embedding_dim': 8,
   'unknown_index': 39}},
 'numerical_scaled': ['bd_clean_scaled',
  'registration_tenure_days_scaled',
  'avg_payment_plan_days_scaled',
  'avg_actual_amount_paid_sca

## Verification

Reload the saved splits from disk and re-check shapes, churn-rate consistency, and null-freeness before treating this pipeline as done.

In [9]:
for name in ["train", "val", "test"]:
    check = pd.read_parquet(os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet"))
    n_nulls = check.drop(columns=["msno"]).isna().sum().sum()
    print(
        f"{name:6s} shape={check.shape}  churn_rate={check['is_churn'].mean():.4f}  "
        f"fwd_rev_59d_median={check['fwd_rev_59d'].median():.0f}  total_nulls={n_nulls}"
    )

train  shape=(1127119, 31)  churn_rate=0.7360  fwd_rev_59d_median=99  total_nulls=0
val    shape=(241526, 31)  churn_rate=0.7360  fwd_rev_59d_median=99  total_nulls=0
test   shape=(241526, 31)  churn_rate=0.7360  fwd_rev_59d_median=99  total_nulls=0
